# istat-delitti-denunciati - notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- le SQL sono la fonte di verità: i check numerici devono essere letti alla luce di quello che dichiarano
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit

In [ ]:
import re
import yaml
import pandas as pd
from pathlib import Path

# --- Unici parametri da impostare manualmente ---
METRICA       = "numero_denunce"  # colonna numerica principale nel mart
METRICA_CLEAN = "numero_denunce"  # colonna corrispondente nel clean

# --- Anchor: usa il path del notebook se disponibile (VSCode), altrimenti cwd ---
try:
    _start = Path(__vsc_ipynb_file__).resolve().parent  # VSCode Jupyter
except NameError:
    _start = Path.cwd().resolve()

# Walk up finché non troviamo dataset.yml
candidate_dir = None
for probe in [_start, *_start.parents]:
    if (probe / "dataset.yml").exists():
        candidate_dir = probe
        break

if candidate_dir is None:
    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")

cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())

SLUG       = cfg["dataset"]["name"]
ANNO_RUN   = cfg["dataset"]["years"][-1]
MART_TABLE = cfg["mart"]["tables"][0]["name"]
ENCODING   = cfg.get("clean", {}).get("read", {}).get("encoding", "utf-8")
DELIM      = cfg.get("clean", {}).get("read", {}).get("delim", ",")
HEADER     = cfg.get("clean", {}).get("read", {}).get("header", True)
SKIP       = cfg.get("clean", {}).get("read", {}).get("skip", 0)

DI_ROOT   = (candidate_dir / cfg["root"]).resolve()
RAW_DIR   = DI_ROOT / "data" / "raw"   / SLUG / str(ANNO_RUN)
CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)
MART_DIR  = DI_ROOT / "data" / "mart"  / SLUG / str(ANNO_RUN)
SQL_DIR   = candidate_dir / "sql"

print(f"slug      : {SLUG}")
print(f"anno_run  : {ANNO_RUN}")
print(f"mart table: {MART_TABLE}")
print(f"encoding  : {ENCODING}  delim: {repr(DELIM)}")
print(f"header    : {HEADER}  skip: {SKIP}")

## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.
Leggile prima di interpretare i check numerici.

In [ ]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()

## 1. Raw

Verifica che il file raw esista, sia leggibile con i parametri dichiarati in `dataset.yml` e abbia il numero di righe atteso. Per questo candidate il raw DBnomics e un CSV wide; la lettura usa le colonne dichiarate in `clean.read.columns` per evitare inferenze miste tra stringhe e numeri.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.xlsx")) + sorted(RAW_DIR.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

read_columns = cfg.get("clean", {}).get("read", {}).get("columns")
read_kwargs = {"encoding": ENCODING, "sep": DELIM}

if raw_path.suffix == ".parquet":
    raw_full = pd.read_parquet(raw_path)
elif raw_path.suffix in (".csv", ".tsv"):
    if read_columns:
        raw_full = pd.read_csv(raw_path, header=0 if HEADER else None, **read_kwargs)
    else:
        header_row = 0 if HEADER else None
        raw_full = pd.read_csv(raw_path, header=header_row, skiprows=SKIP, **read_kwargs)
elif raw_path.suffix == ".xlsx":
    raw_full = pd.read_excel(raw_path, header=0 if HEADER else None, skiprows=SKIP)

N_RAW = len(raw_full)
print(f"Righe raw   : {N_RAW}")
print(f"Colonne raw : {len(raw_full.columns)}")
print(list(raw_full.columns)[:8])
raw_full.head(3)


## 2. Clean

Verifica schema e null. I null su colonne `try_cast` sono attesi se il raw contiene valori non parsabili.
Il confronto righe raw vs clean mostra quante righe sono state filtrate dal `WHERE` della clean.sql.

In [ ]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)

In [ ]:
print(f"Righe raw wide     : {N_RAW}")
print(f"Righe clean long   : {N_CLEAN}")
print("Nota: raw e clean non sono confrontabili 1:1 perche clean.sql converte il CSV wide DBnomics in formato long.")
print()

print("Null per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")


## 3. Mart

Verifica unicità sulle chiavi del GROUP BY, anni presenti, null e consistenza dei totali con il clean.

In [ ]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)

In [ ]:
# mart.sql non aggrega: la chiave naturale attesa e anno + territorio + codice_reato.
expected_keys = ["anno", "codice_territorio", "codice_reato"]
missing_keys = [key for key in expected_keys if key not in mart.columns]
if missing_keys:
    raise AssertionError(f"Chiavi attese non presenti nel mart: {missing_keys}")

dups = mart.duplicated(subset=expected_keys).sum()
print(f"Chiavi attese: {expected_keys}")
print(f"Duplicati: {dups}")
assert dups == 0, "FAIL: duplicati sulle chiavi attese del mart"
print("OK: nessun duplicato sulle chiavi attese.")


In [ ]:
if "anno" in mart.columns:
    anni_mart = sorted(mart["anno"].unique())
    print(f"Anni nel mart ({len(anni_mart)}): {anni_mart}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")

In [ ]:
if METRICA in mart.columns and METRICA_CLEAN in clean.columns:
    tot_mart  = mart[METRICA].sum()
    tot_clean = clean[METRICA_CLEAN].sum()
    delta_pct = abs(tot_mart - tot_clean) / tot_clean * 100 if tot_clean != 0 else float("nan")
    print(f"Totale mart  ({METRICA})      : {tot_mart:,.2f}")
    print(f"Totale clean ({METRICA_CLEAN}): {tot_clean:,.2f}")
    print(f"Delta %: {delta_pct:.4f}%")
    assert delta_pct < 0.01, f"FAIL: delta {delta_pct:.2f}% -- verificare la pipeline"
    print("OK: i totali coincidono.")
else:
    print(f"Colonne non trovate -- aggiornare METRICA ('{METRICA}') e METRICA_CLEAN ('{METRICA_CLEAN}').")

In [ ]:
PERIMETRO = "Italia, indicatore CRIMEN, autore totale, delitti durante l'anno di riferimento; serie 2010-2015 in partizione runtime 2015"

print(f"Slug            : {SLUG}")
print(f"Anno run        : {ANNO_RUN}")
print(f"Tabella mart    : {MART_TABLE}")
print(f"Metrica mart    : {METRICA}")
print(f"Metrica clean   : {METRICA_CLEAN}")
print(f"Perimetro       : {PERIMETRO}")